In [1]:
import pandas as pd
import numpy as np
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'

print("All imports successful ✅")

All imports successful ✅


In [2]:
import subprocess

# Install GEOparse to download GEO datasets
subprocess.run(['pip', 'install', 'GEOparse', '-q'])

import GEOparse

print("Downloading GSE72094...")
print("This is ~400 LUAD patients from Shedden et al.")
print("May take 2-3 minutes...")

gse = GEOparse.get_GEO(geo="GSE72094", 
                        destdir=f'{base}/data/external/',
                        silent=True)

print(f"\nDownload complete!")
print(f"Number of samples: {len(gse.gsms)}")
print(f"Platform: {list(gse.gpls.keys())}")

This is ~400 LUAD patients from Shedden et al.
May take 2-3 minutes...

Download complete!
Number of samples: 442
Platform: ['GPL15048']


In [3]:
# Extract expression matrix and clinical data
print("Extracting expression and survival data...")

# Get all sample data
gsm_data = {}
clinical_data = {}

for gsm_name, gsm in gse.gsms.items():
    # Expression data
    if gsm.table is not None and len(gsm.table) > 0:
        gsm_data[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']
    
    # Clinical metadata
    metadata = gsm.metadata
    clinical_data[gsm_name] = {
        'title': metadata.get('title', [''])[0],
        'characteristics': metadata.get('characteristics_ch1', [])
    }

print(f"Samples with expression data: {len(gsm_data)}")

# Build expression matrix
expr_ext = pd.DataFrame(gsm_data).T
print(f"Raw expression matrix: {expr_ext.shape}")
print(f"First 5 gene IDs: {list(expr_ext.columns[:5])}")

# Check clinical metadata for one sample
print(f"\nExample clinical characteristics:")
first_sample = list(clinical_data.keys())[0]
for c in clinical_data[first_sample]['characteristics']:
    print(f"  {c}")

Extracting expression and survival data...
Samples with expression data: 442
Raw expression matrix: (442, 60607)
First 5 gene IDs: ['AFFX-BioB-5_at', 'AFFX-BioB-M_at', 'AFFX-BioB-3_at', 'AFFX-BioC-5_at', 'AFFX-BioC-3_at']

Example clinical characteristics:
  barcode_microarray: @52070900778263100909406332059157
  filename_microarray: @52070900778263100909406332059157_1_1.CEL
  rna_barin: 8.5
  patient_id: K074
  gender: F
  age_at_diagnosis: 88
  race: NA
  spanish_hispanic: NA
  smoking_status: Missing
  vital_status: NA
  survival_time_in_days: NA
  Stage: NA
  kras_status: WT
  egfr_status: Mut
  stk11_status: WT
  tp53_status: Mut
  egfr_aa_mut: L858LR
  tp53_aa_mut: F149fs*32


In [4]:
# Extract survival data from all samples
survival_records = []

for gsm_name, data in clinical_data.items():
    record = {'sample_id': gsm_name}
    for c in data['characteristics']:
        if ':' in c:
            key, val = c.split(':', 1)
            record[key.strip()] = val.strip()
    survival_records.append(record)

survival_df = pd.DataFrame(survival_records).set_index('sample_id')

print(f"Clinical data shape: {survival_df.shape}")
print(f"Columns: {list(survival_df.columns)}")
print(f"\nVital status values: {survival_df['vital_status'].value_counts().to_dict()}")
print(f"Survival time sample: {survival_df['survival_time_in_days'].head()}")

# Filter to patients with complete survival data
survival_df = survival_df[
    (survival_df['vital_status'] != 'NA') &
    (survival_df['survival_time_in_days'] != 'NA') &
    (survival_df['vital_status'].notna()) &
    (survival_df['survival_time_in_days'].notna())
]

print(f"\nPatients with complete survival data: {len(survival_df)}")
print(f"Vital status: {survival_df['vital_status'].value_counts().to_dict()}")

Clinical data shape: (442, 22)
Columns: ['barcode_microarray', 'filename_microarray', 'rna_barin', 'patient_id', 'gender', 'age_at_diagnosis', 'race', 'spanish_hispanic', 'smoking_status', 'vital_status', 'survival_time_in_days', 'Stage', 'kras_status', 'egfr_status', 'stk11_status', 'tp53_status', 'egfr_aa_mut', 'tp53_aa_mut', 'kras_aa_mut', 'stk11_aa_mut', 'stk11_aa_inherited', 'tp53_aa_inherited']

Vital status values: {'Alive': 298, 'Dead': 122, 'NA': 22}
Survival time sample: sample_id
GSM1854797      NA
GSM1854798    1249
GSM1854799    1057
GSM1854800    1025
GSM1854801     895
Name: survival_time_in_days, dtype: object

Patients with complete survival data: 398
Vital status: {'Alive': 285, 'Dead': 113}


In [5]:
# Get platform annotation to convert probe IDs to gene symbols
print("Extracting platform annotation...")

gpl = gse.gpls['GPL15048']
print(f"Platform table shape: {gpl.table.shape}")
print(f"Platform columns: {list(gpl.table.columns)}")
print(f"\nFirst 3 rows:")
print(gpl.table.head(3))

Extracting platform annotation...
Platform table shape: (60607, 5)
Platform columns: ['ID', 'GB_LIST', 'EntrezGeneID', 'GeneSymbol', 'SPOT_ID']

First 3 rows:
               ID GB_LIST EntrezGeneID GeneSymbol         SPOT_ID
0  AFFX-BioB-3_at     NaN          NaN        NaN  AFFX-BioB-3_at
1  AFFX-BioB-5_at     NaN          NaN        NaN  AFFX-BioB-5_at
2  AFFX-BioB-M_at     NaN          NaN        NaN  AFFX-BioB-M_at


In [6]:
# Build probe to gene symbol mapping
probe_to_gene = gpl.table.set_index('ID')['GeneSymbol']

# Remove probes with no gene symbol
probe_to_gene = probe_to_gene.dropna()
probe_to_gene = probe_to_gene[probe_to_gene != '']

print(f"Total probes: {len(gpl.table)}")
print(f"Probes with gene symbols: {len(probe_to_gene)}")
print(f"Example mappings:")
print(probe_to_gene.head(10))

# Filter expression matrix to probes with gene symbols
expr_ext_filtered = expr_ext[[c for c in expr_ext.columns if c in probe_to_gene.index]]
print(f"\nExpression after filtering to annotated probes: {expr_ext_filtered.shape}")

# Rename columns from probe IDs to gene symbols
expr_ext_filtered.columns = [probe_to_gene[c] for c in expr_ext_filtered.columns]
print(f"After renaming to gene symbols: {expr_ext_filtered.shape}")

# Handle duplicate gene symbols — keep mean across probes
expr_ext_filtered = expr_ext_filtered.astype(float)
expr_ext_filtered = expr_ext_filtered.T.groupby(level=0).mean().T
print(f"After averaging duplicate probes: {expr_ext_filtered.shape}")
print(f"Example genes: {list(expr_ext_filtered.columns[:5])}")

Total probes: 60607
Probes with gene symbols: 41024
Example mappings:
ID
merck2-A18658_at          INSR
merck2-AA004316_a_at     HSDL2
merck2-AA010083_a_at      TPM4
merck2-AA011007_at         GSN
merck2-AA011429_at       CPSF6
merck2-AA021034_at       LTB4R
merck2-AA024853_at       ITSN1
merck2-AA024853_x_at     ITSN1
merck2-AA025001_at      ZNF398
merck2-AA025385_x_at     RPL23
Name: GeneSymbol, dtype: object

Expression after filtering to annotated probes: (442, 41024)
After renaming to gene symbols: (442, 41024)
After averaging duplicate probes: (442, 22115)
Example genes: ['A1BG', 'A1BG-AS1', 'A1CF', 'A2LD1', 'A2M']


In [7]:
# Load our feature columns
feature_cols = json.load(open(f'{base}/models/final/feature_columns_final.json'))

print(f"Our model features: {len(feature_cols)}")
print(f"External dataset genes: {len(expr_ext_filtered.columns)}")

# Check overlap
overlap = set(feature_cols).intersection(set(expr_ext_filtered.columns))
missing = set(feature_cols) - set(expr_ext_filtered.columns)

print(f"Overlapping features: {len(overlap)}")
print(f"Missing features:     {len(missing)}")
print(f"Coverage:             {len(overlap)/len(feature_cols)*100:.1f}%")

if len(missing) > 0:
    print(f"\nMissing features (first 10): {list(missing)[:10]}")

Our model features: 107
External dataset genes: 22115
Overlapping features: 47
Missing features:     30
Coverage:             43.9%

Missing features (first 10): ['T cells CD4 naive', 'T cells follicular helper', 'CLEC18A', 'stage_Stage III', 'Monocytes', 'B cells memory', 'Dendritic cells resting', 'T cells CD4 memory activated', 'Macrophages M0', 'NK cells activated']


In [9]:
# Load immune column names for reference
immune = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)

# Categorise missing features
missing_immune   = [f for f in missing if f in list(immune.columns)]
missing_clinical = [f for f in missing if f in ['age', 'gender',
                    'stage_Stage II', 'stage_Stage III', 'stage_Stage IV']]
missing_genes    = [f for f in missing if f not in missing_immune
                    and f not in missing_clinical]

print(f"Missing immune features:   {len(missing_immune)}")
print(f"Missing clinical features: {len(missing_clinical)}")
print(f"Missing expression genes:  {len(missing_genes)}")
print(f"\nMissing genes: {missing_genes}")
print(f"\nMissing clinical: {missing_clinical}")

Missing immune features:   22
Missing clinical features: 5
Missing expression genes:  3

Missing genes: ['CLEC18A', 'LOC441869', 'CMAH']

Missing clinical: ['stage_Stage III', 'stage_Stage IV', 'gender', 'stage_Stage II', 'age']


In [10]:
# Categorise missing features
missing_immune = [f for f in missing if f in list(immune.columns)]
missing_clinical = [f for f in missing if f in ['age', 'gender', 
                    'stage_Stage II', 'stage_Stage III', 'stage_Stage IV']]
missing_genes = [f for f in missing if f not in missing_immune 
                 and f not in missing_clinical]

print(f"Missing immune features:   {len(missing_immune)}")
print(f"Missing clinical features: {len(missing_clinical)}")
print(f"Missing expression genes:  {len(missing_genes)}")
print(f"\nMissing genes: {missing_genes}")
print(f"\nMissing clinical: {missing_clinical}")

Missing immune features:   22
Missing clinical features: 5
Missing expression genes:  3

Missing genes: ['CLEC18A', 'LOC441869', 'CMAH']

Missing clinical: ['stage_Stage III', 'stage_Stage IV', 'gender', 'stage_Stage II', 'age']


In [11]:
from sklearn.svm import NuSVR
from sklearn.preprocessing import normalize
from scipy.optimize import nnls

# ── Step 1: Fill 3 missing genes with zero ───────────────────────
# These genes aren't on this microarray platform
# Filling with 0 (mean after scaling) is standard practice
for gene in missing_genes:
    expr_ext_filtered[gene] = 0.0
print(f"Filled missing genes with 0: {missing_genes}")

# ── Step 2: Extract clinical features from survival_df ───────────
# Align to patients with survival data
expr_ext_filtered = expr_ext_filtered.loc[
    expr_ext_filtered.index.isin(survival_df.index)]
survival_df = survival_df.loc[survival_df.index.isin(expr_ext_filtered.index)]

print(f"\nAligned patients: {len(expr_ext_filtered)}")

# Age
age_ext = pd.to_numeric(survival_df['age_at_diagnosis'], errors='coerce').fillna(
    pd.to_numeric(survival_df['age_at_diagnosis'], errors='coerce').median())

# Gender
gender_ext = (survival_df['gender'] == 'M').astype(float)

# Stage dummies
def parse_stage(s):
    s = str(s).upper().strip()
    if 'IA' in s or 'IB' in s or s == 'I':
        return 'Stage I'
    elif 'IIA' in s or 'IIB' in s or s == 'II':
        return 'Stage II'
    elif 'IIIA' in s or 'IIIB' in s or s == 'III':
        return 'Stage III'
    elif 'IV' in s:
        return 'Stage IV'
    else:
        return 'Unknown'

stage_parsed = survival_df['Stage'].apply(parse_stage)
print(f"\nStage distribution:")
print(stage_parsed.value_counts())

stage_II_ext  = (stage_parsed == 'Stage II').astype(float)
stage_III_ext = (stage_parsed == 'Stage III').astype(float)
stage_IV_ext  = (stage_parsed == 'Stage IV').astype(float)

clinical_ext = pd.DataFrame({
    'age':            age_ext.values,
    'gender':         gender_ext.values,
    'stage_Stage II': stage_II_ext.values,
    'stage_Stage III':stage_III_ext.values,
    'stage_Stage IV': stage_IV_ext.values
}, index=survival_df.index)

print(f"\nClinical features shape: {clinical_ext.shape}")
print(f"Any NaN: {clinical_ext.isna().any().any()}")

Filled missing genes with 0: ['CLEC18A', 'LOC441869', 'CMAH']

Aligned patients: 398

Stage distribution:
Stage
Unknown    398
Name: count, dtype: int64

Clinical features shape: (398, 5)
Any NaN: False


In [12]:
# Check raw stage values
print("Raw Stage values (first 20):")
print(survival_df['Stage'].value_counts().head(20))
print(f"\nUnique values: {survival_df['Stage'].unique()[:10]}")

Raw Stage values (first 20):
Stage
1A    150
1B     99
2B     49
3A     41
2A     18
3B     16
4      15
1       5
NA      5
Name: count, dtype: int64

Unique values: ['4' '2B' '1B' '1A' '1' '3B' '3A' '2A' 'NA']


In [13]:
def parse_stage(s):
    s = str(s).upper().strip()
    if s in ['NA', 'NAN', '']:
        return 'Unknown'
    elif s.startswith('1') or s == 'I':
        return 'Stage I'
    elif s.startswith('2') or s == 'II':
        return 'Stage II'
    elif s.startswith('3') or s == 'III':
        return 'Stage III'
    elif s.startswith('4') or s == 'IV':
        return 'Stage IV'
    else:
        return 'Unknown'

stage_parsed = survival_df['Stage'].apply(parse_stage)
print(f"Stage distribution:")
print(stage_parsed.value_counts())

# Rebuild clinical features
stage_II_ext  = (stage_parsed == 'Stage II').astype(float)
stage_III_ext = (stage_parsed == 'Stage III').astype(float)
stage_IV_ext  = (stage_parsed == 'Stage IV').astype(float)

clinical_ext = pd.DataFrame({
    'age':             age_ext.values,
    'gender':          gender_ext.values,
    'stage_Stage II':  stage_II_ext.values,
    'stage_Stage III': stage_III_ext.values,
    'stage_Stage IV':  stage_IV_ext.values
}, index=survival_df.index)

print(f"\nClinical features shape: {clinical_ext.shape}")
print(f"Any NaN: {clinical_ext.isna().any().any()}")
print(f"\nStage II patients:  {stage_II_ext.sum():.0f}")
print(f"Stage III patients: {stage_III_ext.sum():.0f}")
print(f"Stage IV patients:  {stage_IV_ext.sum():.0f}")

Stage distribution:
Stage
Stage I      254
Stage II      67
Stage III     57
Stage IV      15
Unknown        5
Name: count, dtype: int64

Clinical features shape: (398, 5)
Any NaN: False

Stage II patients:  67
Stage III patients: 57
Stage IV patients:  15


In [14]:
# Load LM22
lm22 = pd.read_csv(f'{base}/data/external/LM22.txt', sep='\t', index_col=0)

# Find common genes between external expression and LM22
common_lm22 = lm22.index.intersection(expr_ext_filtered.columns)
print(f"LM22 genes:              {len(lm22.index)}")
print(f"External dataset genes:  {len(expr_ext_filtered.columns)}")
print(f"Overlap with LM22:       {len(common_lm22)} ({len(common_lm22)/len(lm22.index)*100:.1f}%)")

expr_lm22 = expr_ext_filtered[common_lm22]
lm22_common = lm22.loc[common_lm22]

# Run DIY CIBERSORT on external data
def run_cibersort_single(patient_expr, lm22_matrix):
    expr_linear = (2 ** patient_expr.values) - 1
    expr_linear = np.clip(expr_linear, 0, 1e6)
    lm22_linear = (2 ** lm22_matrix.values) - 1
    lm22_linear = np.clip(lm22_linear, 0, 1e6)
    lm22_norm   = normalize(lm22_linear, axis=0)
    expr_norm   = normalize(expr_linear.reshape(1, -1))[0]
    best_nu     = 0.5
    best_error  = np.inf
    for nu in [0.25, 0.5, 0.75]:
        try:
            svr = NuSVR(nu=nu, kernel='linear', C=1.0)
            svr.fit(lm22_norm, expr_norm)
            pred  = lm22_norm @ svr.coef_[0]
            error = np.mean((pred - expr_norm) ** 2)
            if error < best_error:
                best_error = error
                best_nu    = nu
        except:
            continue
    try:
        svr = NuSVR(nu=best_nu, kernel='linear', C=1.0)
        svr.fit(lm22_norm, expr_norm)
        raw_weights = svr.coef_[0]
    except:
        raw_weights = np.zeros(lm22_matrix.shape[1])
    clipped = np.maximum(raw_weights, 0)
    if clipped.sum() == 0:
        clipped, _ = nnls(lm22_norm, expr_norm)
        clipped = np.maximum(clipped, 0)
    total = clipped.sum()
    final = clipped / total if total > 0 else np.ones(len(clipped)) / len(clipped)
    return dict(zip(lm22_matrix.columns, final))

from datetime import datetime
print(f"\nRunning DIY CIBERSORT on {len(expr_lm22)} external patients...")
print(f"Start: {datetime.now().strftime('%H:%M:%S')}")

results_ext = {}
for i, patient_id in enumerate(expr_lm22.index):
    results_ext[patient_id] = run_cibersort_single(
        expr_lm22.loc[patient_id], lm22_common)
    if (i+1) % 50 == 0 or i == 0:
        print(f"  {i+1}/{len(expr_lm22)} done [{datetime.now().strftime('%H:%M:%S')}]")

immune_ext = pd.DataFrame(results_ext).T
print(f"\nImmune features shape: {immune_ext.shape}")
print(f"Row sums (should be 1.0): min={immune_ext.sum(axis=1).min():.4f} max={immune_ext.sum(axis=1).max():.4f}")
print(f"Macrophages M2 mean: {immune_ext['Macrophages M2'].mean():.4f}")

LM22 genes:              547
External dataset genes:  22118
Overlap with LM22:       508 (92.9%)

Running DIY CIBERSORT on 398 external patients...
Start: 00:47:13
  1/398 done [00:47:13]
  50/398 done [00:47:15]
  100/398 done [00:47:17]
  150/398 done [00:47:18]
  200/398 done [00:47:20]
  250/398 done [00:47:22]
  300/398 done [00:47:23]
  350/398 done [00:47:25]

Immune features shape: (398, 22)
Row sums (should be 1.0): min=1.0000 max=1.0000
Macrophages M2 mean: 0.0842


In [15]:
from sklearn.preprocessing import StandardScaler
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

# Load our Cox-selected gene lists
cox_expr   = json.load(open(f'{base}/models/final/feature_columns_final.json'))
expr_genes   = [f for f in cox_expr if f in expr_ext_filtered.columns and 
                f not in immune.columns and 
                f not in ['age','gender','stage_Stage II','stage_Stage III','stage_Stage IV']]

# Reload our training feature columns in exact order
feature_cols = json.load(open(f'{base}/models/final/feature_columns_final.json'))

# Build external feature matrix in exact same column order as training
X_ext = pd.DataFrame(index=expr_lm22.index)

for col in feature_cols:
    if col in expr_ext_filtered.columns:
        X_ext[col] = expr_ext_filtered[col].values
    elif col in immune_ext.columns:
        X_ext[col] = immune_ext[col].values
    elif col in clinical_ext.columns:
        X_ext[col] = clinical_ext[col].values
    else:
        X_ext[col] = 0.0  # missing genes filled with 0

print(f"External feature matrix: {X_ext.shape}")
print(f"Any NaN: {X_ext.isna().any().any()}")
print(f"Columns match training: {list(X_ext.columns) == feature_cols}")

# Build survival labels
y_ext = np.array(
    [(vs == 'Dead', float(t)) 
     for vs, t in zip(survival_df['vital_status'], 
                      survival_df['survival_time_in_days'])],
    dtype=[('event', bool), ('time', float)]
)

print(f"\nExternal patients: {len(y_ext)}")
print(f"Events (deaths):   {y_ext['event'].sum()} ({y_ext['event'].mean()*100:.1f}%)")
print(f"Survival range:    {y_ext['time'].min():.0f} to {y_ext['time'].max():.0f} days")

External feature matrix: (398, 77)
Any NaN: False
Columns match training: False

External patients: 398
Events (deaths):   113 (28.4%)
Survival range:    3 to 2077 days


In [16]:
# Check what's missing
print(f"Training features: {len(feature_cols)}")
print(f"External features: {len(X_ext.columns)}")
print(f"\nMissing from external:")
for col in feature_cols:
    if col not in X_ext.columns:
        print(f"  {col}")

Training features: 107
External features: 77

Missing from external:


In [17]:
print(f"X_ext shape: {X_ext.shape}")
print(f"Duplicate columns: {X_ext.columns.duplicated().sum()}")
print(f"Feature cols duplicates: {len(feature_cols) - len(set(feature_cols))}")
print(f"\nFirst 5 cols: {list(X_ext.columns[:5])}")
print(f"Last 5 cols: {list(X_ext.columns[-5:])}")

# Check if all feature_cols are in X_ext
in_ext = [col in X_ext.columns for col in feature_cols]
print(f"\nAll feature_cols in X_ext: {all(in_ext)}")
print(f"Missing count: {sum(not x for x in in_ext)}")

X_ext shape: (398, 77)
Duplicate columns: 0
Feature cols duplicates: 30

First 5 cols: ['LINGO2', 'EPGN', 'DKK1', 'CD109', 'LOC441869']
Last 5 cols: ['age', 'gender', 'stage_Stage II', 'stage_Stage III', 'stage_Stage IV']

All feature_cols in X_ext: True
Missing count: 0


In [18]:
# The issue: same gene appears in both expression AND dysregulation top lists
# e.g. LINGO2 is in both expr top 50 and dysreg top 30
# When building X_ext, pandas deduplicates column names

# Solution: add stream suffix to distinguish them
# Reload original training data to rebuild with suffixes

expr_train   = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg_train = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune_train = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical_train = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

age_tr     = clinical_train[['age']].copy()
gender_tr  = (clinical_train['gender'] == 'male').astype(float).to_frame()
stage_tr   = pd.get_dummies(clinical_train['stage_group'], prefix='stage')
stage_tr   = stage_tr.drop(columns=['stage_Stage I'], errors='ignore')
clinical_tr = pd.concat([age_tr, gender_tr, stage_tr], axis=1).astype(float).fillna(0)

y_train = np.array(
    [(bool(e), t) for e, t in zip(clinical_train['event'], clinical_train['survival_time'])],
    dtype=[('event', bool), ('time', float)]
)

# Reselect top genes with suffixes
from lifelines import CoxPHFitter

times  = y_train['time']
events = y_train['event']

# Top 50 expression genes
cox_pvals_expr = {}
for gene in expr_train.columns:
    try:
        df_tmp = pd.DataFrame({'T': times, 'E': events, 'gene': expr_train[gene].values})
        cph = CoxPHFitter()
        cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
        cox_pvals_expr[gene] = cph.summary['p'].values[0]
    except:
        cox_pvals_expr[gene] = 1.0
top_expr_genes = list(pd.Series(cox_pvals_expr).nsmallest(50).index)

# Top 30 dysregulation genes
cox_pvals_dysreg = {}
for gene in dysreg_train.columns:
    try:
        df_tmp = pd.DataFrame({'T': times, 'E': events, 'gene': dysreg_train[gene].values})
        cph = CoxPHFitter()
        cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
        cox_pvals_dysreg[gene] = cph.summary['p'].values[0]
    except:
        cox_pvals_dysreg[gene] = 1.0
top_dysreg_genes = list(pd.Series(cox_pvals_dysreg).nsmallest(30).index)

print(f"Top expr genes: {len(top_expr_genes)}")
print(f"Top dysreg genes: {len(top_dysreg_genes)}")
print(f"Overlap between streams: {len(set(top_expr_genes).intersection(set(top_dysreg_genes)))}")

Top expr genes: 50
Top dysreg genes: 30
Overlap between streams: 30


In [21]:
# Clinical has 484, others have 478
# Align properly using common patients

expr_train_raw   = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg_train_raw = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune_train_raw = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical_raw     = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

# Find common patients across all
common_patients = (expr_train_raw.index
                   .intersection(dysreg_train_raw.index)
                   .intersection(immune_train_raw.index)
                   .intersection(clinical_raw.index))

print(f"Common patients: {len(common_patients)}")

# Subset all to common patients
expr_top_aligned     = expr_train_raw.loc[common_patients, top_expr_genes].copy()
dysreg_top_aligned   = dysreg_train_raw.loc[common_patients, top_dysreg_genes].copy()
immune_top_aligned   = immune_train_raw.loc[common_patients].copy()
clinical_top_aligned = clinical_raw.loc[common_patients].copy()

# Add suffixes
expr_top_aligned.columns   = [f"{g}_expr" for g in top_expr_genes]
dysreg_top_aligned.columns = [f"{g}_dysreg" for g in top_dysreg_genes]

# Clinical features
age_a     = clinical_top_aligned[['age']].copy()
gender_a  = (clinical_top_aligned['gender'] == 'male').astype(float).to_frame()
stage_a   = pd.get_dummies(clinical_top_aligned['stage_group'], prefix='stage')
stage_a   = stage_a.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features_aligned = pd.concat([age_a, gender_a, stage_a], axis=1).astype(float).fillna(0)

# Build full matrix
X_train_full = pd.concat([
    expr_top_aligned,
    dysreg_top_aligned,
    immune_top_aligned,
    clinical_features_aligned
], axis=1).fillna(0)

# Survival labels
y_train = np.array(
    [(bool(e), t) for e, t in zip(
        clinical_top_aligned['event'],
        clinical_top_aligned['survival_time'])],
    dtype=[('event', bool), ('time', float)])

print(f"Feature matrix: {X_train_full.shape}")
print(f"Duplicate cols: {X_train_full.columns.duplicated().sum()}")
print(f"Any NaN:        {X_train_full.isna().any().any()}")
print(f"Patients:       {len(y_train)}")

# Scale and train
scaler_new     = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler_new.fit_transform(X_train_full),
    columns=X_train_full.columns,
    index=X_train_full.index)

model_new = GradientBoostingSurvivalAnalysis(
    n_estimators=200, learning_rate=0.05, max_depth=2,
    min_samples_split=20, min_samples_leaf=10,
    subsample=0.8, random_state=42)
model_new.fit(X_train_scaled, y_train)

print(f"\nRetrained XGBoost ✅")

Common patients: 478
Feature matrix: (478, 107)
Duplicate cols: 0
Any NaN:        False
Patients:       478

Retrained XGBoost ✅


In [22]:
# Build external feature matrix with same suffix format
expr_ext_top    = expr_ext_filtered[[g for g in top_expr_genes 
                                      if g in expr_ext_filtered.columns]].copy()
dysreg_ext_top  = pd.DataFrame(0.0, index=expr_ext_filtered.index, 
                                columns=top_dysreg_genes)

# Add suffixes
expr_ext_top.columns   = [f"{g}_expr" for g in expr_ext_top.columns]

# For dysregulation — GSE72094 doesn't have GTEx z-scores
# Fill with 0 (conservative — no dysregulation info available)
dysreg_ext_cols = [f"{g}_dysreg" for g in top_dysreg_genes]
dysreg_ext_df   = pd.DataFrame(0.0, index=expr_lm22.index, columns=dysreg_ext_cols)

# Handle missing expression genes
for gene in top_expr_genes:
    col = f"{gene}_expr"
    if col not in expr_ext_top.columns:
        expr_ext_top[col] = 0.0

# Reorder to match training
expr_ext_top = expr_ext_top[[f"{g}_expr" for g in top_expr_genes]]

# Build full external matrix
X_ext_full = pd.concat([
    expr_ext_top,
    dysreg_ext_df,
    immune_ext,
    clinical_ext
], axis=1).fillna(0)

# Reorder columns to exactly match training
X_ext_full = X_ext_full[X_train_full.columns]

print(f"External feature matrix: {X_ext_full.shape}")
print(f"Columns match training:  {list(X_ext_full.columns) == list(X_train_full.columns)}")
print(f"Any NaN: {X_ext_full.isna().any().any()}")

# Scale using training scaler
X_ext_scaled = pd.DataFrame(
    scaler_new.transform(X_ext_full),
    columns=X_ext_full.columns,
    index=X_ext_full.index)

# Predict risk scores
risk_scores_ext = model_new.predict(X_ext_scaled)

# Evaluate
ci_ext = concordance_index_censored(
    y_ext['event'].astype(bool),
    y_ext['time'],
    risk_scores_ext
)[0]

print(f"\n{'='*45}")
print(f"EXTERNAL VALIDATION RESULT (GSE72094)")
print(f"{'='*45}")
print(f"Cohort:          GSE72094 (Shedden et al.)")
print(f"Patients:        {len(y_ext)}")
print(f"Events (deaths): {y_ext['event'].sum()} ({y_ext['event'].mean()*100:.1f}%)")
print(f"C-index:         {ci_ext:.3f}")
print(f"{'='*45}")
print(f"\nComparison:")
print(f"  TCGA training C-index:    0.711")
print(f"  GSE72094 external C-index: {ci_ext:.3f}")

External feature matrix: (398, 107)
Columns match training:  True
Any NaN: False

EXTERNAL VALIDATION RESULT (GSE72094)
Cohort:          GSE72094 (Shedden et al.)
Patients:        398
Events (deaths): 113 (28.4%)
C-index:         0.636

Comparison:
  TCGA training C-index:    0.711
  GSE72094 external C-index: 0.636


In [ ]:
import json
import os

os.makedirs(f'{base}/outputs/results', exist_ok=True)

external_results = {
    "cohort": "GSE72094",
    "reference": "Shedden et al.",
    "platform": "GPL15048 (Affymetrix Merck)",
    "n_patients": 398,
    "n_events": int(y_ext['event'].sum()),
    "event_rate": round(float(y_ext['event'].mean()), 3),
    "c_index": round(float(ci_ext), 3),
    "notes": [
        "Dysregulation stream filled with zeros (no GTEx reference available)",
        "3 genes missing from platform (CLEC18A, LOC441869, CMAH) filled with 0",
        "LM22 gene coverage: 92.9% (508/547 genes)",
        "Different microarray platform from training — domain shift expected"
    ],
    "comparison": {
        "TCGA_training": 0.711,
        "GSE72094_external": round(float(ci_ext), 3)
    }
}

with open(f'{base}/outputs/results/external_validation.json', 'w') as f:
    json.dump(external_results, f, indent=2)

print("Saved: outputs/results/external_validation.json")
print(f"\nExternal validation complete ✅")
print(f"C-index: {ci_ext:.3f} on {len(y_ext)} unseen patients")

In [23]:
import json
import os

os.makedirs(f'{base}/outputs/results', exist_ok=True)

external_results = {
    "cohort": "GSE72094",
    "reference": "Shedden et al.",
    "platform": "GPL15048 (Affymetrix Merck)",
    "n_patients": 398,
    "n_events": int(y_ext['event'].sum()),
    "event_rate": round(float(y_ext['event'].mean()), 3),
    "c_index": round(float(ci_ext), 3),
    "notes": [
        "Dysregulation stream filled with zeros (no GTEx reference available)",
        "3 genes missing from platform (CLEC18A, LOC441869, CMAH) filled with 0",
        "LM22 gene coverage: 92.9% (508/547 genes)",
        "Different microarray platform from training — domain shift expected"
    ],
    "comparison": {
        "TCGA_training": 0.711,
        "GSE72094_external": round(float(ci_ext), 3)
    }
}

with open(f'{base}/outputs/results/external_validation.json', 'w') as f:
    json.dump(external_results, f, indent=2)

print("Saved: outputs/results/external_validation.json")
print(f"\nExternal validation complete ✅")
print(f"C-index: {ci_ext:.3f} on {len(y_ext)} unseen patients")

Saved: outputs/results/external_validation.json

External validation complete ✅
C-index: 0.636 on 398 unseen patients
